# MoM 3D Plate - Tetrahedral Element Interactions 

Two tetrahedron test case. Decompose tetrahedron into 4 facets. 

## Import Packages

In [1]:
using LinearAlgebra  # provides linear algebra toolas 
using StaticArrays   # provides statically typed arrays 
using HCubature      # provides adaptive numerical integration 
using ForwardDiff    # provides forward automatic differentiation 

using Test           # provides testing functionality 

using Plots          # provides plotting functionality 

function hcubature_count(f, a, b; kws...)
    count = 0
    i = hcubature(a, b; kws...) do x
        count += 1
        # display(count)
        f(x)
    end
    return (i..., count)
end;

const Point2D = SVector{2,Float64};
const Point3D = SVector{3,Float64};

const myPoint2D = SVector{2,Real}; # more generic to allow for dual numbders to be created by ForwardDiff
const myPoint3D = SVector{3,Real};

include("mesh_struct_definitions.jl");

<b>Definition of a triangular facet in 3D (goes into seperate file)<b>

In [21]:
Base.@kwdef struct Facet2D 
    r1::Point3D                                # first vertex of the facet  
    r2::Point3D                                # second vertex of the facet
    r3::Point3D                                # third vertex of the facet
    tau1::Point3D = normalize(r2 - r1)         # direction of first edge   
    tau2::Point3D = normalize(r3 - r2)         # direction of second edge
    tau3::Point3D = normalize(r1 - r3)         # direction of third edge
    area::Float64 = .5*norm(cross(tau1,tau2))  # area of triangle 
    nF::Point3D = normalize(cross(tau1,tau2))  # normal to face 
    # n1::Point2D = normalize(cross(tau1,nF))    # normal on first edge   
    # n2::Point2D = normalize(cross(tau2,nF))    # normal on second edge
    # n3::Point2D = normalize(cross(tau3,nF))    # normal on third edge
    Xmat = SMatrix{3,3,Float64, 9}(r1[1], r2[1], r3[1], r1[2], r2[2], r3[2], 1., 1., 1.) 
    rhs  = SMatrix{3,3,Float64, 9}(1I) 
    Emat = SMatrix{3,3,Float64, 9}(Xmat\rhs);
end

# https://docs.julialang.org/en/v1/manual/methods/#Function-like-objects
function (facet2D::Facet2D)(r)
    Emat = facet2D.Emat 
    phi = zeros(3)
    phi = [ Emat[1,i]*r[1] + Emat[2,i]*r[2] + Emat[3,i] for i=1:3]   
    return phi 
end

"""
Integrates a scalar function `f(x::Vector)` over a 3D triangular facet 
"""
function integrate_tri(f, facet; rtol=1e-8, atol=1e-12)
    p1 = facet.r1; p2 = facet.r2; p3 = facet.r3;  
    # Calculate the cross product of the two spanning vectors
    v1 = p2 - p1
    v2 = p3 - p1
    cross_prod = cross(v1, v2)
    
    # 2 * Area of the triangle is the norm of the cross product
    twice_area = norm(cross_prod)
    
    # The transformed integrand mapping [0,1]² to the triangle
    # uv[1] is 'u', uv[2] is 'v'
    integrand = function(uv)
        u, v = uv[1], uv[2]
        
        # Parametric coordinate on the 3D triangle plane
        point = p1 + u * (p2 - p1) + u * v * (p3 - p2)
        
        # f(point) multiplied by the Jacobian factor: 2 * Area * u
        return f(point) * twice_area * u
    end
    
    # Integrate over the unit square [0, 1] × [0, 1]
    val, err = hcubature(integrand, [0.0, 0.0], [1.0, 1.0]; rtol=rtol, atol=atol)
    return val
end; 

## Section 1: Introduction 

(requires more text). 

Assume $P_{\alpha}$ ($P_{\beta}$) to be a tetrahedron bounded by 4 triangular facets $F_{\alpha} \in P_{\alpha}$ ($F_{\beta} \in P_{\beta}$). We wish to compute the 6D volume-volume interaction integral given by  

$$
P_{\alpha \beta} = \int_{P_{\alpha}} \int_{P_{\beta}} \frac{1}{\| {\mathbf r} - {\mathbf r}' \|} \, 
d\Omega' \, d\Omega 
$$

in two ways (cfr. variational form of the grad-div equation for the magnetization using linear shape on tetrahedral elements): 
1. numerically using hcubature, as a double nesting of 3D integrals. We call hcubature separately to compute the inner integral over $P_{\beta}$ and the outer integral $P_{\alpha}$. We are aware of this aproach being computationally inefficient, especially in computing self-term interactions for which $\alpha = \beta$. We are, however, merely interested in the generating reference results. Computational efficiency is therefore not a concern here. 
2. analytically using Euler integration theorem. We reduce the dimension of the inner and outer integral for 3D (volume of tetrahdron) to 2D (surface of a triangular facet) and arrive at a set of 4*4 = 16 4D integrals. We evaluate these integrals numerically as a double nesting of 2D integrals. This approach remains computationally inefficient and further reduction in dimension remains required. 

Given that 

$$
\nabla' \cdot \frac{\mathbf{r} - \mathbf{r}'}{\| \mathbf{r} - \mathbf{r}' \|} = \frac{-2}{\| \mathbf{r} - \mathbf{r}' \|}
$$ 

we obtain that 

$$
P_{\alpha \beta} = - \frac{1}{2} \int_{P_{\alpha}} \int_{P_{\beta}} \nabla' \cdot \frac{\mathbf{r} - \mathbf{r}'}{\| \mathbf{r} - \mathbf{r}' \|} d\Omega' \, d\Omega = - \frac{1}{2} \int_{P_{\alpha}} \int_{F_{\beta} \in P_{\beta}} \frac{(\mathbf{r} - \mathbf{r}') \cdot \mathbf{n}'}{\| \mathbf{r} - \mathbf{r}' \|} dS' \, d\Omega \, . 
$$

By interchanging the order of integration, we obtain 

$$
- \frac{1}{2} \int_{P_{\alpha}} \int_{F_{\beta} \in P_{\beta}} \frac{(\mathbf{r} - \mathbf{r}') \cdot \mathbf{n}'}{\| \mathbf{r} - \mathbf{r}' \|} \, dS' \, d\Omega = 
- \frac{1}{2} \int_{F_{\beta} \in P_{\beta}} \mathbf{n}' \cdot \int_{P_{\alpha}} \frac{\mathbf{r} - \mathbf{r}'}{\| \mathbf{r} - \mathbf{r}' \|} \, d\Omega \, dS' \, . 
$$

We apply Euler Integration Theorem for homogeneous functions on the inner volume integral over $P_{\alpha}$ (insert reference here) to obtain

$$
- \frac{1}{2} \int_{F_{\beta} \in P_{\beta}} \mathbf{n}' \cdot \int_{P_{\alpha}} \frac{\mathbf{r} - \mathbf{r}'}{\| \mathbf{r} - \mathbf{r}' \|} \, d\Omega \, dS' \, = 
- \frac{1}{6} \int_{F_{\alpha} \in P_{\alpha}} \int_{F_{\beta} \in P_{\beta}} \frac{\left[(\mathbf{r} - \mathbf{r}') \cdot \mathbf{n} \right] \, \left[(\mathbf{r} - \mathbf{r}') \cdot \mathbf{n}'\right]}{\| \mathbf{r} - \mathbf{r}' \|} \, dS' \, dS
$$

We thus obtain that the 6D volume-volume interaction integral $I_{\alpha \beta}$ is given as the sum of $4*4 = 16$ 4D surface-surface interaction integral, or that 

$$
\int_{P_{\alpha}} \int_{P_{\beta}} \frac{1}{\| {\mathbf r} - {\mathbf r}' \|} \, 
d\Omega' \, d\Omega = - \frac{1}{6} \int_{F_{\alpha} \in P_{\alpha}} \int_{F_{\beta} \in P_{\beta}} \frac{\left[(\mathbf{r} - \mathbf{r}') \cdot \mathbf{n} \right] \, \left[(\mathbf{r} - \mathbf{r}') \cdot \mathbf{n}'\right]}{\| \mathbf{r} - \mathbf{r}' \|} \, dS' \, dS \, .  
$$

<b>Exercises</b>
1. profile the number of integrand evaluations by replacing calls to <i>hcubature</i> by calls to <i>hcubature_count</i>. The latter stores the number of integrand evaluations in the third element of the output array; 
2. make a graphical representation of how $P_{\alpha\beta}$ varries with the distance between $P_{\alpha}$ and $P_{\beta}$. Connect this representation with results from literature on the singular Reisz kernel; 

## Section 2: Define reference element ($P_{\alpha}$) and $x$-shifted reference element ($P_{\beta}$)

We choose $P_{\alpha}$ to be the reference element. We choose $P_{\beta}$ a shift along the $x$-axis of $P_{\beta}$ (other shifts and other transformations will be considered later). 

In [41]:
#..Reference element - P_{\alpha} 

#....4 Points of reference element 
refr1 = Point3D(0.,0.,0.); refr2 = Point3D(1.,0.,0.);
refr3 = Point3D(0.,1.,0.); refr4 = Point3D(0.,0.,1.);

#....Volume of reference element
refvol  = evalVol(refr1,refr2,refr3,refr4)
refEmat = genBasis(refr1,refr2,refr3,refr4)
indices = Vector(1:12) 
refElem = Elem3DLin(refr1,refr2,refr3,refr4,indices,refEmat,refvol)

#....4 Faces of reference element
refF1 = Facet2D(r1 = refr1, r2 = refr3, r3 = refr2)
refF2 = Facet2D(r1 = refr1, r2 = refr2, r3 = refr4)
refF3 = Facet2D(r1 = refr1, r2 = refr4, r3 = refr3)
refF4 = Facet2D(r1 = refr2, r2 = refr3, r3 = refr4)
refF  = [refF1, refF2, refF3, refF4]
println("normals on reference element")
display([refF1.nF refF2.nF refF3.nF refF4.nF]) 

#..Shifted reference element - P_{\beta} 

#....4 Points of shifted reference element 
shift = Point3D(5.,0.,0.)
primer1 = Point3D(0.,0.,0.)+shift; primer2 = Point3D(1.,0.,0.) + shift; 
primer3 = Point3D(0.,1.,0.)+shift; primer4 = Point3D(0.,0.,1.) + shift;

#....Volume of shifted reference element
primevol  = evalVol(primer1,primer2,primer3,primer4)
primeEmat = genBasis(primer1,primer2,primer3,primer4)
primeElem = Elem3DLin(primer1,primer2,primer3,primer4,indices,primeEmat,primevol)

#....4 Faces of shifted reference element
primeF1 = Facet2D(r1 = primer1, r2 = primer3, r3 = primer2)
primeF2 = Facet2D(r1 = primer1, r2 = primer2, r3 = primer4)
primeF3 = Facet2D(r1 = primer1, r2 = primer4, r3 = primer3)
primeF4 = Facet2D(r1 = primer2, r2 = primer3, r3 = primer4)
primeF  = [primeF1, primeF2, primeF3, primeF4]
println("normals on shifted reference element")
display([primeF1.nF primeF2.nF primeF3.nF primeF4.nF]) 

normals on reference element


3×4 SMatrix{3, 4, Float64, 12} with indices SOneTo(3)×SOneTo(4):
  0.0   0.0  -1.0  0.57735
  0.0  -1.0   0.0  0.57735
 -1.0   0.0   0.0  0.57735

normals on shifted reference element


3×4 SMatrix{3, 4, Float64, 12} with indices SOneTo(3)×SOneTo(4):
  0.0   0.0  -1.0  0.57735
  0.0  -1.0   0.0  0.57735
 -1.0   0.0   0.0  0.57735

## Section 3: Test of Elementary Functions on Elements and Facets

In [42]:
v = vcat(refElem.Emat[:,1],refElem.Emat[:,2],refElem.Emat[:,3])

12-element MVector{12, Float64} with indices SOneTo(12):
 -1.0
 -1.0
 -1.0
  1.0
  1.0
  0.0
  0.0
  0.0
  0.0
  1.0
  0.0
  0.0

In [43]:
evalBasis(primer4,primeElem)

4-element SVector{4, Float64} with indices SOneTo(4):
 -8.881784197001252e-16
  8.881784197001252e-16
  0.0
  1.0

In [44]:
integrand = x->x[1]+x[2]+x[3] 
integrate_tet(integrand,refElem)

0.12499999999999994

## Section 4: 6D Volume Interaction Integral 

In [45]:
inner_integral_tet(rd,sourceElem) = integrate_tet(rs->kernel(rd,rs), sourceElem)

outer_integral_tet(destElem, sourceElem) = 
    integrate_tet(rd->inner_integral(rd,sourceElem), destElem)

outer_integral_tet (generic function with 1 method)

In [46]:
outer_integral_tet(refElem, primeElem)

0.005555531094601571

In [47]:
normals = [Point3D(0.,0.,-1.), 
           Point3D(-1.,0.,0.), 
           Point3D(0.,-1.,0.), 
           Point3D(1.,1.,1.)/sqrt(3)]
normalsp = deepcopy(normals)
[dot(n,np) for n in normals, np in normalsp] 

4×4 Matrix{Float64}:
  1.0       0.0       0.0      -0.57735
  0.0       1.0       0.0      -0.57735
  0.0       0.0       1.0      -0.57735
 -0.57735  -0.57735  -0.57735   1.0

## Section 5: 4D Surface Interaction Integral 

In [48]:
inner_integrand_tri(rd,rs,destFacet,sourceFacet) = 
    dot(sourceFacet.nF,rd-rs)*dot(destFacet.nF,rd-rs)*kernel(rd,rs)

inner_integral_tri(rd,destFacet,sourceFacet) = 
    integrate_tri(rs->inner_integrand_tri(rd,rs,destFacet,sourceFacet), sourceFacet)

outer_integral_tri(destFacet,sourceFacet) = 
    integrate_tri(rd->inner_integral_tri(rd,destFacet,sourceFacet), destFacet)

outer_integral_tri (generic function with 1 method)

In [49]:
out = [outer_integral_tri(refFi, primeFj) for refFi in refF, primeFj in primeF] 

4×4 Matrix{Float64}:
  0.0         -0.00550622   0.0825835  -0.0898833
 -0.00550622   0.0          0.0825835  -0.0898833
 -0.0827487   -0.0827487    1.24452    -1.32812
  0.0761112    0.0761112   -1.16078     1.24994

In [50]:
(-1/6)*sum(out)

0.005555532350441779

## Experimental Section: Integrate over Source Domain, Differentiate w.r.t. and Integrate Again 

Possibly valuable in generate alternative reference solutions or experiment with other types of integrals. 

**Part-(1/3)** We compute the scalar function 

$$ 
I_{inner}({\mathbf r}) = \int_{P_{\beta}} \frac{\phi_1({\mathbf r})}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' \, . 
$$ 

In [24]:
basis_fct_beta(rs)      = evalBasis(rs,primeElement)[1]
inv_distance(rd, rs)    = 1/norm(rd - rs)
inner_integrand(rd, rs) = inv_distance(rd, rs)*basis_fct_beta(rs)
inner_integral(rd)      = hcubature(rs->inner_integrand(rd,myPoint3D(rs[1],rs[2],rs[3])), (5.,5.,5.), (6.,6.,6.))[1]

inner_integral (generic function with 1 method)

In [25]:
rd = myPoint3D(.5,.5,.5)
inner_integral(rd)

-1.2105114429821546

**Part-(2/3)** We compute the vector function 

$$
\text{grad}_{\mathbf{r}} I_{inner}({\mathbf r}) = \nabla_{\mathbf{r}} I_{inner}({\mathbf r}) = \nabla_{\mathbf{r}} \int_{P_{\beta}} \frac{\phi_1({\mathbf r})}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' \, .
$$


In [26]:
dinner_integral(rd) = ForwardDiff.gradient(rd->inner_integral(myPoint3D(rd[1],rd[2],rd[3])), rd)

dinner_integral (generic function with 1 method)

In [27]:
dinner_integral(rd)

3-element SVector{3, Float64} with indices SOneTo(3):
 -0.08057267789194224
 -0.08057267789194222
 -0.08057267789194224

**Part-(3/3)** We compute the scalar  

$$
\int_{P_{\alpha}} \text{grad}_{\mathbf{r}} I_{inner}({\mathbf r}) \, d\Omega = \int_{P_{\alpha}} \nabla_{\mathbf{r}} I_{inner}({\mathbf r}) \, d\Omega = \int_{P_{\alpha}} \nabla_{\mathbf{r}} \int_{P_{\beta}} \frac{\phi_1({\mathbf r})}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' \, d\Omega \, .
$$

In [19]:
outer_integral = hcubature(rd->dinner_integral(myPoint3D(rd[1],rd[2],rd[3])), (0,0,0), (1,1,1))[1]


3-element SVector{3, Float64} with indices SOneTo(3):
 -0.08057337332964216
 -0.08057337332968717
 -0.08057337332995872

All of above in one go. 

In [31]:
a_alphabeta = zeros(4)
b_alphabeta = zeros(4)
c_alphabeta = zeros(4)

for ind = 1:4 
    
    basis_fct_prime(rs)     = evalBasis(rs,primeElement)[ind]
    inv_distance(rd, rs)    = 1/norm(rd - rs)
    inner_integrand(rd, rs) = inv_distance(rd, rs)*basis_fct_prime(rs)
    inner_integral(rd)      = hcubature(rs->inner_integrand(rd,myPoint3D(rs[1],rs[2],rs[3])), (5.,5.,5.), (6.,6.,6.))[1]

    dinner_integral(rd) = ForwardDiff.gradient(rd->inner_integral(myPoint3D(rd[1],rd[2],rd[3])), rd)

    outer_integral = hcubature(rd->dinner_integral(myPoint3D(rd[1],rd[2],rd[3])), (0,0,0), (1,1,1))[1]
 
    a_alphabeta[ind], b_alphabeta[ind], c_alphabeta[ind] = outer_integral
    
end 

In [33]:
a_alpha = refEmat[1,:]; b_alpha = refEmat[2,:]; c_alpha = refEmat[3,:]  

4-element SVector{4, Float64} with indices SOneTo(4):
 -1.0
  0.0
  0.0
  1.0